[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/10-engineering-toolbox/05_cli-linux.ipynb)

*Engineering Toolbox · Notebook 5 of 8*

# The Command Line & Linux

The shell skills every engineer needs, rebuilt in Python: permissions and chmod, globs, paths, argparse CLIs, exit codes and &&/||, wc, sort | uniq -c, grep, head and tail, cut, cron schedules, du, find, log parsing, variable expansion, process trees, signals, disk alerts and shell quoting.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** The shell is a workshop of small tools; pipes are the conveyor belt between them.

**What interviewers really test here:** Navigating and searching files, pipes, permissions, exit codes, environment variables, processes and signals, and writing your own CLI.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Moving around and reading files *(Basics)*
- [ ] 2. Pipes: small tools, combined *(Core)*
- [ ] 3. Writing your own command-line tools *(Core)*
- [ ] 4. Processes, signals and scheduled jobs *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Commands worth knowing

| Need | Command |
| --- | --- |
| Search text in files | `grep -rn "pattern" dir/` |
| Find files | `find . -name "*.py" -mtime -7` |
| Follow a log | `tail -f app.log` |
| Disk usage | `df -h`, `du -sh *` |
| Processes and ports | `ps aux`, `top`, `ss -tlnp` |
| Remote server | `ssh user@host`, `scp file host:/path` |
| HTTP from the terminal | `curl -sS https://api/health` |
| JSON on the command line | `jq '.items[].id' data.json` |

---
# Part 2: Lessons, step by step

## Lesson 1: Moving around and reading files

*Level: Basics*

Servers, containers and CI runners have no mouse — the shell is how you work with them. A path starting with `/` is absolute; others start from the current directory; `~` is your home, `..` the parent.



```python
pwd; ls -la; cd ~/projects/app; cd -          # where am I, what's here, go back
cat config.yaml; less app.log; head -n 20 data.csv; tail -f app.log
wc -l data.csv                                   # how many lines
cp -r src backup/; mv old.txt new.txt; mkdir -p logs/2026; rm -i tmp.txt
ls *.py; ls data/202?-*.csv                      # globs are expanded by the shell
```



Every file has an owner, a group and permissions for owner/group/others: `-rwxr-x---` is `750`. Scripts need `x` to run; secrets should be `600`.

**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q2](#q2), [Q6](#q6), [Q3](#q3), [Q1](#q1), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 2: Pipes: small tools, combined

*Level: Core*

Each Unix tool does one thing and reads standard input; the pipe `|` sends one tool's output into the next. Together they answer real questions in one line:



```python
grep -c " 500 " access.log                         # how many server errors?
cut -d' ' -f1 access.log | sort | uniq -c | sort -rn | head   # top client IPs
grep -rn "TODO" --include="*.py" src/              # find TODOs with line numbers
find logs -name "*.log" -size +1M -mtime +30       # big old logs
du -sh * | sort -h; df -h                          # what's using the disk
```



Every command returns an **exit code** (0 = success; `echo $?` shows it). `a && b` runs b only if a succeeded, `a || b` only if it failed — CI scripts rely on this, and `set -euo pipefail` makes a script stop at the first failure.

**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q4](#q4), [Q7](#q7), [Q9](#q9), [Q8](#q8), [Q11](#q11), [Q15](#q15), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 3: Writing your own command-line tools

*Level: Core*

Python scripts become real tools with `argparse` (built in) or Typer/Click: positional arguments, options with defaults, flags, help text, and a non-zero exit code on error.



```python
import argparse, sys

def main(argv=None):
    p = argparse.ArgumentParser(prog="report", description="Summarise sales files.")
    p.add_argument("inputs", nargs="+", help="CSV files to read")
    p.add_argument("--since", help="only rows on or after this date")
    p.add_argument("--format", choices=["csv", "json"], default="csv")
    p.add_argument("-v", "--verbose", action="count", default=0)
    p.add_argument("--dry-run", action="store_true")
    args = p.parse_args(argv)
    ...
    return 0

if __name__ == "__main__":
    sys.exit(main())

# pyproject.toml → installs a `report` command
[project.scripts]
report = "app.cli:main"
```



The shell does work before your program starts: it splits the line into arguments (respecting quotes) and expands `$VARIABLES`. Quote values with spaces: `"$FILE"`.

**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q20](#q20), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 4: Processes, signals and scheduled jobs

*Level: Advanced*

- `ps aux`, `top`/`htop` show processes; each has a parent. Stop a service's children before the parent.
- `kill` sends **SIGTERM** (please finish up); after a grace period, **SIGKILL** ends it with no cleanup. Handle SIGTERM in long-running code so containers shut down cleanly.
- **cron** runs jobs on a schedule: `minute hour day month weekday`. On Kubernetes it's a CronJob; in Airflow, a schedule.



```python
import signal, sys
def stop(signum, frame):
    finish_current_batch(); sys.exit(0)
signal.signal(signal.SIGTERM, stop)

# crontab -e
*/15 9-17 * * 1-5  /opt/app/.venv/bin/python -m app.sync >> /var/log/sync.log 2>&1
```

**✏️ Your turn:** you now have what you need for [Q19](#q19), [Q10](#q10), [Q16](#q16). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [2]:
import re

import fnmatch

from collections import Counter, defaultdict

import posixpath


<a id="q1"></a>
### Q1. Read ls -l Permissions

🟢 Warm-up · *rwx to octal*

Convert permission strings like `-rwxr-xr--` (type character, then owner, group, others) into `(type, octal)`: type is `'file'`, `'dir'` or `'link'`; each rwx triple is r=4 + w=2 + x=1.

**Examples**

- `entries = ['-rwxr-xr--', 'drwxr-x---', '-rw-r--r--', 'lrwxrwxrwx', '-rw-------']` → `[('file', '754'), ('dir', '750'), ('file', '644'), ('link', '777'), ('file', '600')]`

In [ ]:
def perms(entries):
    """
    Convert permission strings like -rwxr-xr-- (type character, then owner,
    group, others) into (type, octal): type is 'file', 'dir' or 'link'; each rwx
    triple is r=4 + w=2 + x=1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(perms, [
    ((['-rwxr-xr--', 'drwxr-x---', '-rw-r--r--', 'lrwxrwxrwx', '-rw-------'],), [('file', '754'), ('dir', '750'), ('file', '644'), ('link', '777'), ('file', '600')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add up each triple with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Map each character to a bit and use int(..., 2).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Add up each triple with if-statements.

```python
def perms(entries):
    out = []
    for e in entries:
        kind = {"-": "file", "d": "dir", "l": "link"}[e[0]]
        digits = ""
        for i in range(1, 10, 3):
            v = 0
            if e[i] == "r":
                v += 4
            if e[i + 1] == "w":
                v += 2
            if e[i + 2] == "x":
                v += 1
            digits += str(v)
        out.append((kind, digits))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Map each character to a bit and use int(..., 2).

```python
def perms(entries):
    return [({"-": "file", "d": "dir", "l": "link"}[e[0]], format(int("".join("0" if c == "-" else "1" for c in e[1:]), 2), "03o")) for e in entries]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add up each triple with if-statements. That costs O(n) time and O(n) space."

"I can do better: Map each character to a bit and use int(..., 2). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/read-ls-l-permissions)

<a id="q2"></a>
### Q2. Match Shell Globs

🟢 Warm-up · *\*, ? and [abc]*

For each pattern, return the paths it matches. As in the shell, `*` and `?` never match a `/`, so match segment by segment (same number of segments).

**Examples**

- `patterns = ['*.py', 'src/*.py', 'data/202?-*.csv', 'tests/test_[ab]*.py', '*/*/*.json'], paths = ['setup.py', 'src/app.py', 'src/util/io.py', 'data/2024-01.csv', 'data/2025-06.csv', 'data/archive/2023-01.csv', 'tests/test_api.py', 'tests/test_cli.py', 'tests/test_b2.py', 'conf/dev/settings.json']` → `{'*.py': ['setup.py'], 'src/*.py': ['src/app.py'], 'data/202?-*.csv': ['data/2024-01.csv', 'data/2025-06.csv'], 'tests/test_[ab]*.py': ['tests/test_api.py', 'tests/test_b2.py'], '*/*/*.json': ['conf/dev/settings.json']}`

In [ ]:
def globs(patterns, paths):
    """
    For each pattern, return the paths it matches. As in the shell, * and ?
    never match a /, so match segment by segment (same number of segments).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(globs, [
    ((['*.py', 'src/*.py', 'data/202?-*.csv', 'tests/test_[ab]*.py', '*/*/*.json'], ['setup.py', 'src/app.py', 'src/util/io.py', 'data/2024-01.csv', 'data/2025-06.csv', 'data/archive/2023-01.csv', 'tests/test_api.py', 'tests/test_cli.py', 'tests/test_b2.py', 'conf/dev/settings.json']), {'*.py': ['setup.py'], 'src/*.py': ['src/app.py'], 'data/202?-*.csv': ['data/2024-01.csv', 'data/2025-06.csv'], 'tests/test_[ab]*.py': ['tests/test_api.py', 'tests/test_b2.py'], '*/*/*.json': ['conf/dev/settings.json']}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split into segments and fnmatch each.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Translate each pattern to one regex where \* is [^/]\*.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Split into segments and fnmatch each.

```python
def globs(patterns, paths):
    out = {}
    for pat in patterns:
        pp = pat.split("/")
        hits = []
        for path in paths:
            parts = path.split("/")
            if len(parts) == len(pp) and all(fnmatch.fnmatchcase(a, b) for a, b in zip(parts, pp)):
                hits.append(path)
        out[pat] = hits
    return out
```

**Cost:** O(p · n · L) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Translate each pattern to one regex where \* is [^/]\*.

```python
def globs(patterns, paths):
    def rx(pat):
        r = ""
        for ch in re.split(r"(\*|\?|\[[^\]]*\])", pat):
            r += "[^/]*" if ch == "*" else "[^/]" if ch == "?" else ch if ch.startswith("[") else re.escape(ch)
        return re.compile(r)
    return {p: [x for x in paths if rx(p).fullmatch(x)] for p in patterns}
```

**Cost:** O(p · n · L) time, O(L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split into segments and fnmatch each. That costs O(p · n · L) time and O(L) space."

"I can do better: Translate each pattern to one regex where \* is [^/]\*. That's O(p · n · L) time and O(L) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `patterns = ['*.txt'], paths = ['a.txt', 'd/b.txt']`

1. rx() returns re.compile('[^/]\*\\\\.txt')
2. rx() returns re.compile('[^/]\*\\\\.txt')
3. globs() returns {'\*.txt': ['a.txt']}

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/match-shell-globs)

<a id="q3"></a>
### Q3. wc: Lines, Words, Characters

🟢 Warm-up · *Counting text*

Return `(lines, words, characters)` like `wc`: lines counts newline characters, words are runs of non-whitespace, characters counts every character.

**Examples**

- `text = 'ERROR db timeout\nINFO  started\n\nWARN  slow query 812ms\n'` → `(4, 9, 55)`
- `text = 'no newline at end'` → `(0, 4, 17)`

In [ ]:
def wc(text):
    """
    Return (lines, words, characters) like wc: lines counts newline characters,
    words are runs of non-whitespace, characters counts every character.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(wc, [
    (('ERROR db timeout\nINFO  started\n\nWARN  slow query 812ms\n',), (4, 9, 55)),
    (('no newline at end',), (0, 4, 17)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

One pass with a state flag.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`count`, `split` and `len`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** One pass with a state flag.

```python
def wc(text):
    lines = words = 0
    in_word = False
    for ch in text:
        if ch == "\n":
            lines += 1
        if ch.isspace():
            in_word = False
        elif not in_word:
            words += 1
            in_word = True
    return (lines, words, len(text))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `count`, `split` and `len`.

```python
def wc(text):
    return (text.count("\n"), len(text.split()), len(text))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: One pass with a state flag. That costs O(n) time and O(1) space."

"I can do better: `count`, `split` and `len`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/wc-lines-words-characters)

<a id="q4"></a>
### Q4. sort | uniq -c | sort -rn | head

🟢 Warm-up · *The classic count pipeline*

Count each line in a log (e.g. client IPs), and return the top `k` as `(count, line)`, highest count first (ties: line sorted ascending) — exactly what `sort | uniq -c | sort -rn | head -n k` shows.

**Examples**

- `lines = ['10.0.0.5', '10.0.0.9', '10.0.0.5', '192.168.1.20', '10.0.0.9', '10.0.0.5', '172.16.0.2', '192.168.1.20', '10.0.0.9'], k = 3` → `[(3, '10.0.0.5'), (3, '10.0.0.9'), (2, '192.168.1.20')]`

In [ ]:
def top_lines(lines, k):
    """
    Count each line in a log (e.g. client IPs), and return the top k as (count,
    line), highest count first (ties: line sorted ascending) — exactly what sort
    | uniq -c | sort -rn | head -n k shows.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_lines, [
    ((['10.0.0.5', '10.0.0.9', '10.0.0.5', '192.168.1.20', '10.0.0.9', '10.0.0.5', '172.16.0.2', '192.168.1.20', '10.0.0.9'], 3), [(3, '10.0.0.5'), (3, '10.0.0.9'), (2, '192.168.1.20')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort, group adjacent equal lines, then sort counts.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter` and a sort key.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sort, group adjacent equal lines, then sort counts.

```python
def top_lines(lines, k):
    s = sorted(lines)
    groups = []
    i = 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        groups.append((j - i, s[i]))
        i = j
    groups.sort(key=lambda g: (-g[0], g[1]))
    return groups[:k]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter` and a sort key.

```python
def top_lines(lines, k):
    return sorted(((n, l) for l, n in Counter(lines).items()), key=lambda g: (-g[0], g[1]))[:k]
```

**Cost:** O(n + u log u) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort, group adjacent equal lines, then sort counts. That costs O(n log n) time and O(n) space."

"I can do better: `Counter` and a sort key. That's O(n + u log u) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/sort-uniq-c-sort-rn-head)

<a id="q5"></a>
### Q5. grep With Flags

🟢 Warm-up · *-i, -v, -n, -c*

Implement `grep`: return matching lines for a regex `pattern`, honouring flags in `flags`: `i` (ignore case), `v` (invert), `n` (prefix `'line:'` with the 1-based line number), `c` (return only the count).

**Examples**

- `pattern = 'error|warn', flags = 'i', lines = ['INFO boot ok', 'WARN disk 91%', 'ERROR db timeout', 'info error budget ok']` → `['WARN disk 91%', 'ERROR db timeout', 'info error budget ok']`
- `pattern = '^INFO', flags = 'vn', lines = ['INFO boot ok', 'WARN disk 91%', 'ERROR db timeout']` → `['2:WARN disk 91%', '3:ERROR db timeout']`
- `pattern = 'timeout', flags = 'c', lines = ['ERROR db timeout', 'ERROR api timeout', 'INFO ok']` → `2`

In [ ]:
def grep(pattern, flags, lines):
    """
    Implement grep: return matching lines for a regex pattern, honouring flags
    in flags: i (ignore case), v (invert), n (prefix 'line:' with the 1-based
    line number), c (return only the count).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(grep, [
    (('error|warn', 'i', ['INFO boot ok', 'WARN disk 91%', 'ERROR db timeout', 'info error budget ok']), ['WARN disk 91%', 'ERROR db timeout', 'info error budget ok']),
    (('^INFO', 'vn', ['INFO boot ok', 'WARN disk 91%', 'ERROR db timeout']), ['2:WARN disk 91%', '3:ERROR db timeout']),
    (('timeout', 'c', ['ERROR db timeout', 'ERROR api timeout', 'INFO ok']), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with if-statements per flag.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compile once; comprehension with an XOR for -v.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop with if-statements per flag.

```python
def grep(pattern, flags, lines):
    out = []
    for i, line in enumerate(lines, 1):
        if "i" in flags:
            hit = re.search(pattern, line, re.IGNORECASE) is not None
        else:
            hit = re.search(pattern, line) is not None
        if "v" in flags:
            hit = not hit
        if hit:
            out.append(f"{i}:{line}" if "n" in flags else line)
    return len(out) if "c" in flags else out
```

**Cost:** O(n · L) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compile once; comprehension with an XOR for -v.

```python
def grep(pattern, flags, lines):
    rx = re.compile(pattern, re.I if "i" in flags else 0)
    out = [f"{i}:{l}" if "n" in flags else l for i, l in enumerate(lines, 1) if bool(rx.search(l)) != ("v" in flags)]
    return len(out) if "c" in flags else out
```

**Cost:** O(n · L) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with if-statements per flag. That costs O(n · L) time and O(n) space."

"I can do better: Compile once; comprehension with an XOR for -v. That's O(n · L) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/grep-with-flags)

<a id="q6"></a>
### Q6. head and tail

🟢 Warm-up · *Slices of a file*

Run a small pipeline of `head -n N` and `tail -n N` steps (and `tail -n +N`, meaning 'from line N on') over the lines, in order. Return the result.

**Examples**

- `lines = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20], steps = ['head -n 15', 'tail -n 4']` → `[12, 13, 14, 15]`
- `lines = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20], steps = ['tail -n +18']` → `[18, 19, 20]`
- `lines = [1, 2, 3, 4, 5], steps = ['tail -n 10', 'head -n 2']` → `[1, 2]`

In [ ]:
def head_tail(lines, steps):
    """
    Run a small pipeline of head -n N and tail -n N steps (and tail -n +N,
    meaning 'from line N on') over the lines, in order. Return the result.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(head_tail, [
    (([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20], ['head -n 15', 'tail -n 4']), [12, 13, 14, 15]),
    (([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20], ['tail -n +18']), [18, 19, 20]),
    (([1, 2, 3, 4, 5], ['tail -n 10', 'head -n 2']), [1, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Rebuild lists with loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Slicing.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Rebuild lists with loops.

```python
def head_tail(lines, steps):
    cur = list(lines)
    for s in steps:
        cmd, _, n = s.split()
        if cmd == "head":
            new = []
            for i in range(min(int(n), len(cur))):
                new.append(cur[i])
            cur = new
        elif n.startswith("+"):
            cur = [cur[i] for i in range(int(n) - 1, len(cur))]
        else:
            start = max(0, len(cur) - int(n))
            cur = [cur[i] for i in range(start, len(cur))]
    return cur
```

**Cost:** O(s · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Slicing.

```python
def head_tail(lines, steps):
    cur = list(lines)
    for s in steps:
        cmd, _, n = s.split()
        cur = cur[:int(n)] if cmd == "head" else cur[int(n) - 1:] if n[0] == "+" else cur[-int(n):] if int(n) else []
    return cur
```

**Cost:** O(s · n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Rebuild lists with loops. That costs O(s · n) time and O(n) space."

"I can do better: Slicing. That's O(s · n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/head-and-tail)

<a id="q7"></a>
### Q7. cut Columns

🟢 Warm-up · *cut -d, -f2,4*

Return the selected 1-based fields (a field list like `'2,4'` or a range like `'2-4'`, or a mix such as `'1,3-4'`) from each line split on the delimiter, joined back with the delimiter. Missing fields are left out.

**Examples**

- `lines = ['id,name,city,plan', '1,Asha,Pune,pro', '2,Ravi,Delhi', '3,Meera,Goa,basic'], delim = ',', fields = '1,3-4'` → `['id,city,plan', '1,Pune,pro', '2,Delhi', '3,Goa,basic']`

In [ ]:
def cut(lines, delim, fields):
    """
    Return the selected 1-based fields (a field list like '2,4' or a range like
    '2-4', or a mix such as '1,3-4') from each line split on the delimiter,
    joined back with the delimiter. Missing fields are left out.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cut, [
    ((['id,name,city,plan', '1,Asha,Pune,pro', '2,Ravi,Delhi', '3,Meera,Goa,basic'], ',', '1,3-4'), ['id,city,plan', '1,Pune,pro', '2,Delhi', '3,Goa,basic']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Expand the field list with loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set of field numbers and a comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Expand the field list with loops.

```python
def cut(lines, delim, fields):
    wanted = []
    for part in fields.split(","):
        if "-" in part:
            a, b = part.split("-")
            for f in range(int(a), int(b) + 1):
                wanted.append(f)
        else:
            wanted.append(int(part))
    out = []
    for line in lines:
        cols = line.split(delim)
        picked = []
        for i in range(len(cols)):
            if i + 1 in wanted:
                picked.append(cols[i])
        out.append(delim.join(picked))
    return out
```

**Cost:** O(n · f) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set of field numbers and a comprehension.

```python
def cut(lines, delim, fields):
    want = set()
    for part in fields.split(","):
        a, _, b = part.partition("-")
        want |= set(range(int(a), int(b or a) + 1))
    return [delim.join(c for i, c in enumerate(l.split(delim), 1) if i in want) for l in lines]
```

**Cost:** O(n · f) time, O(f) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Expand the field list with loops. That costs O(n · f) time and O(f) space."

"I can do better: A set of field numbers and a comprehension. That's O(n · f) time and O(f) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/cut-columns)

<a id="q8"></a>
### Q8. du -sh: What's Using the Disk?

🟢 Warm-up · *Sizes by folder*

Given file sizes in bytes (path → size), total them by top-level folder (files at the top level count under their own name) and return `(name, human size)` largest first, using `human` (1024-based units).

**Examples**

- `files = {'data/raw/events.parquet': 3221225472, 'data/raw/users.csv': 52428800, 'logs/app.log': 734003200, 'logs/app.log.1': 1073741824, 'src/main.py': 4096, 'README.md': 2048, '.venv/lib/torch.so': 2147483648}` → `[('data', '3.0G'), ('.venv', '2.0G'), ('logs', '1.7G'), ('src', '4.0K'), ('README.md', '2.0K')]`

In [ ]:
def du(files):
    """
    Given file sizes in bytes (path → size), total them by top-level folder
    (files at the top level count under their own name) and return (name, human
    size) largest first, using human (1024-based units).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(du, [
    (({'data/raw/events.parquet': 3221225472, 'data/raw/users.csv': 52428800, 'logs/app.log': 734003200, 'logs/app.log.1': 1073741824, 'src/main.py': 4096, 'README.md': 2048, '.venv/lib/torch.so': 2147483648},), [('data', '3.0G'), ('.venv', '2.0G'), ('logs', '1.7G'), ('src', '4.0K'), ('README.md', '2.0K')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sum into a dict, then sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`defaultdict(int)` and sorted items.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sum into a dict, then sort.

```python
def human(size):
    for unit in ["B", "K", "M", "G", "T"]:
        if size < 1024 or unit == "T":
            return f"{size:.0f}{unit}" if unit == "B" else f"{size:.1f}{unit}"
        size /= 1024


def du(files):
    totals = {}
    for path, size in files.items():
        top = path.split("/")[0]
        totals[top] = totals.get(top, 0) + size
    rows = sorted(totals.items(), key=lambda r: -r[1])
    return [(name, human(size)) for name, size in rows]
```

**Cost:** O(f log f) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `defaultdict(int)` and sorted items.

```python
def human(size):
    for unit in ["B", "K", "M", "G", "T"]:
        if size < 1024 or unit == "T":
            return f"{size:.0f}{unit}" if unit == "B" else f"{size:.1f}{unit}"
        size /= 1024


def du(files):
    t = defaultdict(int)
    for path, size in files.items():
        t[path.partition("/")[0]] += size
    return [(n, human(s)) for n, s in sorted(t.items(), key=lambda r: -r[1])]
```

**Cost:** O(f log f) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sum into a dict, then sort. That costs O(f log f) time and O(d) space."

"I can do better: `defaultdict(int)` and sorted items. That's O(f log f) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/du-sh-whats-using-the-disk)

<a id="q9"></a>
### Q9. find Files by Name, Size and Age

🟢 Warm-up · *find . -name -size -mtime*

Files are `(path, size_kb, age_days)`. Return the paths whose file name matches `name` (a glob), whose size is over `min_kb` and whose age is over `min_days` — e.g. old big logs to clean up.

**Examples**

- `files = [('logs/app-2026-10-01.log', 52000, 57), ('logs/app-2026-11-26.log', 48000, 1), ('logs/tiny.log', 3, 90), ('data/export.csv', 900000, 120), ('logs/archive/app-2026-08-01.log', 61000, 118)], name = '*.log', min_kb = 1024, min_days = 30` → `['logs/app-2026-10-01.log', 'logs/archive/app-2026-08-01.log']`

In [ ]:
def find_files(files, name, min_kb, min_days):
    """
    Files are (path, size_kb, age_days). Return the paths whose file name
    matches name (a glob), whose size is over min_kb and whose age is over
    min_days — e.g. old big logs to clean up.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_files, [
    (([('logs/app-2026-10-01.log', 52000, 57), ('logs/app-2026-11-26.log', 48000, 1), ('logs/tiny.log', 3, 90), ('data/export.csv', 900000, 120), ('logs/archive/app-2026-08-01.log', 61000, 118)], '*.log', 1024, 30), ['logs/app-2026-10-01.log', 'logs/archive/app-2026-08-01.log']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with three conditions.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop with three conditions.

```python
def find_files(files, name, min_kb, min_days):
    out = []
    for path, kb, age in files:
        base = path.split("/")[-1]
        if fnmatch.fnmatch(base, name) and kb > min_kb and age > min_days:
            out.append(path)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def find_files(files, name, min_kb, min_days):
    return [p for p, kb, age in files if fnmatch.fnmatch(posixpath.basename(p), name) and kb > min_kb and age > min_days]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with three conditions. That costs O(n) time and O(n) space."

"I can do better: A comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/find-files-by-name-size-and-age)

<a id="q10"></a>
### Q10. SIGTERM, Then SIGKILL

🟢 Warm-up · *Graceful shutdown*

On shutdown, each process gets SIGTERM and has `grace` seconds to finish (Docker and Kubernetes default to 10 and 30). A process needing longer is killed with SIGKILL and loses its in-flight work. Given shutdown times, return `(exited cleanly, killed)` as sorted name lists.

**Examples**

- `times = {'web': 2.5, 'worker': 45.0, 'scheduler': 0.4, 'exporter': 10.0, 'consumer': 12.0}, grace = 10` → `(['exporter', 'scheduler', 'web'], ['consumer', 'worker'])`

In [ ]:
def shutdown(times, grace):
    """
    On shutdown, each process gets SIGTERM and has grace seconds to finish
    (Docker and Kubernetes default to 10 and 30). A process needing longer is
    killed with SIGKILL and loses its in-flight work. Given shutdown times,
    return (exited cleanly, killed) as sorted name lists.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shutdown, [
    (({'web': 2.5, 'worker': 45.0, 'scheduler': 0.4, 'exporter': 10.0, 'consumer': 12.0}, 10), (['exporter', 'scheduler', 'web'], ['consumer', 'worker'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop.

```python
def shutdown(times, grace):
    clean, killed = [], []
    for name, t in times.items():
        if t <= grace:
            clean.append(name)
        else:
            killed.append(name)
    return (sorted(clean), sorted(killed))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two comprehensions.

```python
def shutdown(times, grace):
    return (sorted(n for n, t in times.items() if t <= grace), sorted(n for n, t in times.items() if t > grace))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n log n) time and O(n) space."

"I can do better: Two comprehensions. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/sigterm-then-sigkill)

<a id="q11"></a>
### Q11. Disk Space Alerts

🟢 Warm-up · *df -h*

Rows are `(filesystem, mount, size_gb, used_gb)`. Return `(mount, use%)` rounded to whole percent for filesystems at or above `warn`%, highest first, marking each `'CRIT'` if at or above `crit`% else `'WARN'`.

**Examples**

- `rows = [('/dev/nvme0n1p2', '/', 100, 61), ('/dev/nvme1n1', '/data', 2000, 1910), ('tmpfs', '/run', 8, 0.1), ('/dev/sdb1', '/var/log', 50, 42.5), ('/dev/sdc1', '/backup', 4000, 3200)], warn = 80, crit = 90` → `[('/data', 96, 'CRIT'), ('/var/log', 85, 'WARN'), ('/backup', 80, 'WARN')]`

In [ ]:
def disk_alerts(rows, warn, crit):
    """
    Rows are (filesystem, mount, size_gb, used_gb). Return (mount, use%) rounded
    to whole percent for filesystems at or above warn%, highest first, marking
    each 'CRIT' if at or above crit% else 'WARN'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(disk_alerts, [
    (([('/dev/nvme0n1p2', '/', 100, 61), ('/dev/nvme1n1', '/data', 2000, 1910), ('tmpfs', '/run', 8, 0.1), ('/dev/sdb1', '/var/log', 50, 42.5), ('/dev/sdc1', '/backup', 4000, 3200)], 80, 90), [('/data', 96, 'CRIT'), ('/var/log', 85, 'WARN'), ('/backup', 80, 'WARN')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute and filter in a loop, then sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A sorted comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute and filter in a loop, then sort.

```python
def disk_alerts(rows, warn, crit):
    out = []
    for _, mount, size, used in rows:
        pct = round(used / size * 100)
        if pct >= warn:
            out.append((mount, pct, "CRIT" if pct >= crit else "WARN"))
    out.sort(key=lambda r: -r[1])
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A sorted comprehension.

```python
def disk_alerts(rows, warn, crit):
    pcts = [(m, round(100 * u / s)) for _, m, s, u in rows]
    return sorted(((m, p, "CRIT" if p >= crit else "WARN") for m, p in pcts if p >= warn), key=lambda r: -r[1])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute and filter in a loop, then sort. That costs O(n log n) time and O(n) space."

"I can do better: A sorted comprehension. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/disk-space-alerts)

<a id="q12"></a>
### Q12. Apply chmod Changes

🟡 Interview standard · *u+x, go-w, a=r*

Start from an octal mode (string like `'644'`) and apply symbolic changes in order: who (`u`, `g`, `o`, `a`, combinable), an operator (`+` add, `-` remove, `=` set exactly) and permissions (`r`, `w`, `x`). Return the mode after each change.

**Examples**

- `mode = '644', changes = ['u+x', 'go-r', 'a+x', 'g=rw', 'o=', 'a-w']` → `['744', '700', '711', '761', '760', '540']`

In [ ]:
def chmod(mode, changes):
    """
    Start from an octal mode (string like '644') and apply symbolic changes in
    order: who (u, g, o, a, combinable), an operator (+ add, - remove, = set
    exactly) and permissions (r, w, x). Return the mode after each change.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(chmod, [
    (('644', ['u+x', 'go-r', 'a+x', 'g=rw', 'o=', 'a-w']), ['744', '700', '711', '761', '760', '540']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep three digits and update each with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Work on a 9-bit integer with masks.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Keep three digits and update each with if-statements.

```python
def chmod(mode, changes):
    digits = [int(c) for c in mode]
    out = []
    for ch in changes:
        m = re.fullmatch(r"([ugoa]+)([+=-])([rwx]*)", ch)
        who, op, what = m.groups()
        if "a" in who:
            who = "ugo"
        bits = 0
        for p in what:
            bits += {"r": 4, "w": 2, "x": 1}[p]
        for w in who:
            i = "ugo".index(w)
            if op == "+":
                digits[i] = digits[i] | bits
            elif op == "-":
                digits[i] = digits[i] & ~bits
            else:
                digits[i] = bits
        out.append("".join(str(d) for d in digits))
    return out
```

**Cost:** O(c) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Work on a 9-bit integer with masks.

```python
def chmod(mode, changes):
    m, out = int(mode, 8), []
    for ch in changes:
        who, op, what = re.fullmatch(r"([ugoa]+)([+=-])([rwx]*)", ch).groups()
        bits = sum({"r": 4, "w": 2, "x": 1}[p] for p in what)
        shifts = [6, 3, 0] if "a" in who else [{"u": 6, "g": 3, "o": 0}[w] for w in who]
        for s in shifts:
            m = m | bits << s if op == "+" else m & ~(bits << s) if op == "-" else (m & ~(7 << s)) | bits << s
        out.append(format(m, "03o"))
    return out
```

**Cost:** O(c) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep three digits and update each with if-statements. That costs O(c) time and O(1) space."

"I can do better: Work on a 9-bit integer with masks. That's O(c) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/apply-chmod-changes)

<a id="q13"></a>
### Q13. Resolve Paths

🟡 Interview standard · *cd, ., .. and ~*

Turn each path into an absolute, normalised path, as the shell would from the current directory `cwd` with home `home`: `~` at the start means home, relative paths start from cwd, `.` and empty parts are dropped, `..` goes up (never above `/`).

**Examples**

- `cwd = '/home/asha/proj', home = '/home/asha', paths = ['src/../tests', './data//raw/', '~/notes.txt', '/var/log/../../etc', '../../../../..', '~']` → `['/home/asha/proj/tests', '/home/asha/proj/data/raw', '/home/asha/notes.txt', '/etc', '/', '/home/asha']`

In [ ]:
def resolve_paths(cwd, home, paths):
    """
    Turn each path into an absolute, normalised path, as the shell would from
    the current directory cwd with home home: ~ at the start means home,
    relative paths start from cwd, . and empty parts are dropped, .. goes up
    (never above /).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(resolve_paths, [
    (('/home/asha/proj', '/home/asha', ['src/../tests', './data//raw/', '~/notes.txt', '/var/log/../../etc', '../../../../..', '~']), ['/home/asha/proj/tests', '/home/asha/proj/data/raw', '/home/asha/notes.txt', '/etc', '/', '/home/asha']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A stack of parts.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`posixpath.join` and `normpath`, then fix the root case.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A stack of parts.

```python
def resolve_paths(cwd, home, paths):
    out = []
    for p in paths:
        if p == "~" or p.startswith("~/"):
            p = home + p[1:]
        if not p.startswith("/"):
            p = cwd + "/" + p
        stack = []
        for part in p.split("/"):
            if part in ("", "."):
                continue
            if part == "..":
                if stack:
                    stack.pop()
            else:
                stack.append(part)
        out.append("/" + "/".join(stack))
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `posixpath.join` and `normpath`, then fix the root case.

```python
def resolve_paths(cwd, home, paths):
    out = []
    for p in paths:
        p = posixpath.join(cwd, home + p[1:] if p == "~" or p.startswith("~/") else p)
        n = posixpath.normpath(p)
        out.append("/" + n.lstrip("/"))
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A stack of parts. That costs O(n · L) time and O(L) space."

"I can do better: `posixpath.join` and `normpath`, then fix the root case. That's O(n · L) time and O(L) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/resolve-paths)

<a id="q14"></a>
### Q14. Build a CLI With argparse

🟡 Interview standard · *Options, flags and arguments*

The tool `report` takes one or more input files, an optional `--since DATE`, `--format` (`csv` or `json`, default csv), a repeatable `-v`/`--verbose` counter and a `--dry-run` flag. Return the parsed settings for each argument list as a dict. The brute force parses by hand; the optimal uses `argparse` (`report_parser()`).

**Examples**

- `argvs = [['sales.csv'], ['a.csv', 'b.csv', '--format', 'json', '-v', '-v'], ['--since', '2026-01-01', '--dry-run', 'x.csv', '--verbose'], ['-vv', 'jan.csv', '--format=json']]` → `[{'inputs': ['sales.csv'], 'since': None, 'format': 'csv', 'verbose': 0, 'dry_run': False}, {'inputs': ['a.csv', 'b.csv'], 'since': None, 'format': 'json', 'verbose': 2, 'dry_run': False}, {'inputs': ['x.csv'], 'since': '2026-01-01', 'format': 'csv', 'verbose': 1, 'dry_run': True}, {'inputs': ['jan.csv'], 'since': None, 'format': 'json', 'verbose': 2, 'dry_run': False}]`

In [ ]:
import argparse

def parse_cli(argvs):
    """
    The tool report takes one or more input files, an optional --since DATE,
    --format (csv or json, default csv), a repeatable -v/--verbose counter and a
    --dry-run flag. Return the parsed settings for each argument list as a dict.
    The brute force parses by hand; the optimal uses argparse (report_parser()).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(parse_cli, [
    (([['sales.csv'], ['a.csv', 'b.csv', '--format', 'json', '-v', '-v'], ['--since', '2026-01-01', '--dry-run', 'x.csv', '--verbose'], ['-vv', 'jan.csv', '--format=json']],), [{'inputs': ['sales.csv'], 'since': None, 'format': 'csv', 'verbose': 0, 'dry_run': False}, {'inputs': ['a.csv', 'b.csv'], 'since': None, 'format': 'json', 'verbose': 2, 'dry_run': False}, {'inputs': ['x.csv'], 'since': '2026-01-01', 'format': 'csv', 'verbose': 1, 'dry_run': True}, {'inputs': ['jan.csv'], 'since': None, 'format': 'json', 'verbose': 2, 'dry_run': False}]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the arguments with a while loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`argparse.ArgumentParser.parse_args`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk the arguments with a while loop.

```python
def parse_cli(argvs):
    out = []
    for argv in argvs:
        r = {"inputs": [], "since": None, "format": "csv", "verbose": 0, "dry_run": False}
        i = 0
        while i < len(argv):
            a = argv[i]
            if a.startswith("--") and "=" in a:
                key, val = a[2:].split("=", 1)
                r[key.replace("-", "_")] = val
            elif a in ("--since", "--format"):
                r[a[2:]] = argv[i + 1]
                i += 1
            elif a == "--dry-run":
                r["dry_run"] = True
            elif a == "--verbose":
                r["verbose"] += 1
            elif a.startswith("-v") and set(a[1:]) == {"v"}:
                r["verbose"] += len(a) - 1
            else:
                r["inputs"].append(a)
            i += 1
        out.append(r)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `argparse.ArgumentParser.parse_args`.

```python
def report_parser():
    p = argparse.ArgumentParser(prog="report")
    p.add_argument("inputs", nargs="+")
    p.add_argument("--since")
    p.add_argument("--format", choices=["csv", "json"], default="csv")
    p.add_argument("-v", "--verbose", action="count", default=0)
    p.add_argument("--dry-run", action="store_true")
    return p


def parse_cli(argvs):
    parser = report_parser()
    return [vars(parser.parse_args(a)) for a in argvs]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the arguments with a while loop. That costs O(n) time and O(n) space."

"I can do better: `argparse.ArgumentParser.parse_args`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/build-a-cli-with-argparse)

<a id="q15"></a>
### Q15. Exit Codes, && and ||

🟡 Interview standard · *How the shell chains commands*

Each command has an exit code (0 = success). In `a && b`, b runs only if the last status is 0; in `a || b`, b runs only if it is non-zero; operators apply left to right. Given a chain like `['test', '&&', 'build', '||', 'notify']` and the exit code of each command, return `(commands that ran, final exit status)`.

**Examples**

- `tokens = ['lint', '&&', 'test', '&&', 'deploy'], codes = {'lint': 0, 'test': 1, 'deploy': 0}` → `(['lint', 'test'], 1)`
- `tokens = ['build', '||', 'notify', '&&', 'cleanup'], codes = {'build': 2, 'notify': 0, 'cleanup': 0}` → `(['build', 'notify', 'cleanup'], 0)`
- `tokens = ['make', '||', 'echo'], codes = {'make': 0, 'echo': 0}` → `(['make'], 0)`

In [ ]:
def chain(tokens, codes):
    """
    Each command has an exit code (0 = success). In a && b, b runs only if the
    last status is 0; in a || b, b runs only if it is non-zero; operators apply
    left to right. Given a chain like ['test', '&&', 'build', '||', 'notify']
    and the exit code of each command, return (commands that ran, final exit
    status).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(chain, [
    ((['lint', '&&', 'test', '&&', 'deploy'], {'lint': 0, 'test': 1, 'deploy': 0}), (['lint', 'test'], 1)),
    ((['build', '||', 'notify', '&&', 'cleanup'], {'build': 2, 'notify': 0, 'cleanup': 0}), (['build', 'notify', 'cleanup'], 0)),
    ((['make', '||', 'echo'], {'make': 0, 'echo': 0}), (['make'], 0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the list with an index.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Iterate over (operator, command) pairs.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk the list with an index.

```python
def chain(tokens, codes):
    ran = [tokens[0]]
    status = codes[tokens[0]]
    i = 1
    while i < len(tokens):
        op, cmd = tokens[i], tokens[i + 1]
        if (op == "&&" and status == 0) or (op == "||" and status != 0):
            ran.append(cmd)
            status = codes[cmd]
        i += 2
    return (ran, status)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Iterate over (operator, command) pairs.

```python
def chain(tokens, codes):
    ran, status = [tokens[0]], codes[tokens[0]]
    for op, cmd in zip(tokens[1::2], tokens[2::2]):
        if (status == 0) == (op == "&&"):
            ran.append(cmd)
            status = codes[cmd]
    return (ran, status)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the list with an index. That costs O(n) time and O(n) space."

"I can do better: Iterate over (operator, command) pairs. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/exit-codes-and)

<a id="q16"></a>
### Q16. When Does the Cron Job Run?

🟡 Interview standard · *minute hour day month weekday*

Given a cron expression (fields: minute 0–59, hour 0–23, day of month 1–31, month 1–12, weekday 0–6 with 0 = Sunday; supporting `*`, lists, ranges and `/step`) and a start time `'YYYY-MM-DD HH:MM'`, return the next `n` run times strictly after the start. (For simplicity, a run needs both the day-of-month and weekday fields to match.)

**Examples**

- `expr = '*/15 9-17 * * 1-5', start = '2026-11-27 17:40', n = 4` → `['2026-11-27 17:45', '2026-11-30 09:00', '2026-11-30 09:15', '2026-11-30 09:30']`
- `expr = '0 2 1 * *', start = '2026-11-27 10:00', n = 2` → `['2026-12-01 02:00', '2027-01-01 02:00']`

In [ ]:
import datetime as dt

def cron_next(expr, start, n):
    """
    Given a cron expression (fields: minute 0–59, hour 0–23, day of month 1–31,
    month 1–12, weekday 0–6 with 0 = Sunday; supporting *, lists, ranges and
    /step) and a start time 'YYYY-MM-DD HH:MM', return the next n run times
    strictly after the start. (For simplicity, a run needs both the day-of-month
    and weekday fields to match.)
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cron_next, [
    (('*/15 9-17 * * 1-5', '2026-11-27 17:40', 4), ['2026-11-27 17:45', '2026-11-30 09:00', '2026-11-30 09:15', '2026-11-30 09:30']),
    (('0 2 1 * *', '2026-11-27 10:00', 2), ['2026-12-01 02:00', '2027-01-01 02:00']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step forward one minute at a time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Skip whole days and hours that can't match.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Step forward one minute at a time.

```python
def cron_field(expr, lo, hi):
    out = set()
    for part in expr.split(","):
        rng, _, step = part.partition("/")
        if rng == "*":
            a, b = lo, hi
        elif "-" in rng:
            a, b = map(int, rng.split("-"))
        else:
            a = b = int(rng)
        out |= set(range(a, b + 1, int(step or 1)))
    return out


def cron_next(expr, start, n):
    mins, hours, doms, months, dows = (cron_field(f, lo, hi) for f, (lo, hi) in zip(expr.split(), [(0, 59), (0, 23), (1, 31), (1, 12), (0, 6)]))
    t = dt.datetime.strptime(start, "%Y-%m-%d %H:%M")
    out = []
    while len(out) < n:
        t += dt.timedelta(minutes=1)
        if t.minute in mins and t.hour in hours and t.day in doms and t.month in months and (t.weekday() + 1) % 7 in dows:
            out.append(t.strftime("%Y-%m-%d %H:%M"))
    return out
```

**Cost:** O(minutes searched) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Skip whole days and hours that can't match.

```python
def cron_field(expr, lo, hi):
    out = set()
    for part in expr.split(","):
        rng, _, step = part.partition("/")
        if rng == "*":
            a, b = lo, hi
        elif "-" in rng:
            a, b = map(int, rng.split("-"))
        else:
            a = b = int(rng)
        out |= set(range(a, b + 1, int(step or 1)))
    return out


def cron_next(expr, start, n):
    mins, hours, doms, months, dows = (sorted(cron_field(f, lo, hi)) for f, (lo, hi) in zip(expr.split(), [(0, 59), (0, 23), (1, 31), (1, 12), (0, 6)]))
    t0 = dt.datetime.strptime(start, "%Y-%m-%d %H:%M")
    day, out = t0.date(), []
    while len(out) < n:
        if day.day in doms and day.month in months and (day.weekday() + 1) % 7 in dows:
            for h in hours:
                for m in mins:
                    t = dt.datetime(day.year, day.month, day.day, h, m)
                    if t > t0 and len(out) < n:
                        out.append(t.strftime("%Y-%m-%d %H:%M"))
        day += dt.timedelta(days=1)
    return out
```

**Cost:** O(days + matches) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step forward one minute at a time. That costs O(minutes searched) time and O(1) space."

"I can do better: Skip whole days and hours that can't match. That's O(days + matches) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/when-does-the-cron-job-run)

<a id="q17"></a>
### Q17. Parse an Access Log

🟡 Interview standard · *Regex on real log lines*

Parse nginx access-log lines with the regex `LOG` and return `{'by_class': counts of 2xx/3xx/4xx/5xx, 'top_path': most requested path (ties alphabetical), 'bytes': total bytes sent, 'bad_lines': lines that didn't parse}`.

**Examples**

- `lines = ['10.0.0.5 - - [27/Nov/2026:10:00:01 +0530] "GET /api/orders HTTP/1.1" 200 512 "-" "curl/8.4"', '10.0.0.9 - - [27/Nov/2026:10:00:02 +0530] "POST /api/login HTTP/1.1" 401 98 "-" "Mozilla/5.0"', '10.0.0.5 - - [27/Nov/2026:10:00:03 +0530] "GET /api/orders HTTP/1.1" 200 530 "-" "curl/8.4"', 'garbage line', '10.0.0.7 - - [27/Nov/2026:10:00:04 +0530] "GET /health HTTP/1.1" 200 2 "-" "kube-probe"', '10.0.0.9 - - [27/Nov/2026:10:00:05 +0530] "GET /api/orders HTTP/1.1" 502 157 "-" "Mozilla/5.0"', '10.0.0.8 - - [27/Nov/2026:10:00:06 +0530] "GET /old HTTP/1.1" 301 0 "-" "Mozilla/5.0"']` → `{'by_class': {'2xx': 3, '3xx': 1, '4xx': 1, '5xx': 1}, 'top_path': '/api/orders', 'bytes': 1299, 'bad_lines': 1}`

In [ ]:
LOG = re.compile(r'^(\S+) \S+ \S+ \[[^\]]+\] "(\S+) (\S+) [^"]*" (\d{3}) (\d+)')

def access_log(lines):
    """
    Parse nginx access-log lines with the regex LOG and return {'by_class':
    counts of 2xx/3xx/4xx/5xx, 'top_path': most requested path (ties
    alphabetical), 'bytes': total bytes sent, 'bad_lines': lines that didn't
    parse}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(access_log, [
    ((['10.0.0.5 - - [27/Nov/2026:10:00:01 +0530] "GET /api/orders HTTP/1.1" 200 512 "-" "curl/8.4"', '10.0.0.9 - - [27/Nov/2026:10:00:02 +0530] "POST /api/login HTTP/1.1" 401 98 "-" "Mozilla/5.0"', '10.0.0.5 - - [27/Nov/2026:10:00:03 +0530] "GET /api/orders HTTP/1.1" 200 530 "-" "curl/8.4"', 'garbage line', '10.0.0.7 - - [27/Nov/2026:10:00:04 +0530] "GET /health HTTP/1.1" 200 2 "-" "kube-probe"', '10.0.0.9 - - [27/Nov/2026:10:00:05 +0530] "GET /api/orders HTTP/1.1" 502 157 "-" "Mozilla/5.0"', '10.0.0.8 - - [27/Nov/2026:10:00:06 +0530] "GET /old HTTP/1.1" 301 0 "-" "Mozilla/5.0"'],), {'by_class': {'2xx': 3, '3xx': 1, '4xx': 1, '5xx': 1}, 'top_path': '/api/orders', 'bytes': 1299, 'bad_lines': 1}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Dicts and if-statements per line.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Counters over the parsed tuples.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Dicts and if-statements per line.

```python
def access_log(lines):
    by_class, paths, total, bad = {}, {}, 0, 0
    for line in lines:
        m = LOG.match(line)
        if not m:
            bad += 1
            continue
        _, _, path, status, size = m.groups()
        cls = status[0] + "xx"
        by_class[cls] = by_class.get(cls, 0) + 1
        paths[path] = paths.get(path, 0) + 1
        total += int(size)
    top = sorted(paths, key=lambda p: (-paths[p], p))[0]
    return {"by_class": dict(sorted(by_class.items())), "top_path": top, "bytes": total, "bad_lines": bad}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Counters over the parsed tuples.

```python
def access_log(lines):
    parsed = [m.groups() for m in map(LOG.match, lines) if m]
    paths = Counter(p for _, _, p, _, _ in parsed)
    return {"by_class": dict(sorted(Counter(s[0] + "xx" for *_, s, _ in parsed).items())),
            "top_path": min(paths, key=lambda p: (-paths[p], p)), "bytes": sum(int(b) for *_, b in parsed),
            "bad_lines": len(lines) - len(parsed)}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Dicts and if-statements per line. That costs O(n) time and O(n) space."

"I can do better: Counters over the parsed tuples. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/parse-an-access-log)

<a id="q18"></a>
### Q18. Expand Shell Variables

🟡 Interview standard · *\$VAR, \${VAR} and \${VAR:-default}*

Expand `$NAME`, `${NAME}` and `${NAME:-default}` (default used when the variable is unset or empty) in each string using `env`; unknown variables become empty. Text in single quotes is not expanded (keep it, without the quotes).

**Examples**

- `strings = ['$HOME/projects/${APP}_logs', '${PORT:-8080}', 'db=${DB_HOST:-localhost}:${DB_PORT:-5432}', "'$HOME stays literal' but $USER does not", '${EMPTY:-fallback}${MISSING}end'], env = {'HOME': '/home/asha', 'APP': 'billing', 'DB_HOST': 'db.internal', 'USER': 'asha', 'EMPTY': ''}` → `['/home/asha/projects/billing_logs', '8080', 'db=db.internal:5432', '$HOME stays literal but asha does not', 'fallbackend']`

In [ ]:
def expand(strings, env):
    """
    Expand $NAME, ${NAME} and ${NAME:-default} (default used when the variable
    is unset or empty) in each string using env; unknown variables become empty.
    Text in single quotes is not expanded (keep it, without the quotes).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(expand, [
    ((['$HOME/projects/${APP}_logs', '${PORT:-8080}', 'db=${DB_HOST:-localhost}:${DB_PORT:-5432}', "'$HOME stays literal' but $USER does not", '${EMPTY:-fallback}${MISSING}end'], {'HOME': '/home/asha', 'APP': 'billing', 'DB_HOST': 'db.internal', 'USER': 'asha', 'EMPTY': ''}), ['/home/asha/projects/billing_logs', '8080', 'db=db.internal:5432', '$HOME stays literal but asha does not', 'fallbackend']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A character-by-character scanner.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Split on single-quoted parts; re.sub elsewhere.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A character-by-character scanner.

```python
def expand(strings, env):
    out = []
    for s in strings:
        r, i = "", 0
        while i < len(s):
            c = s[i]
            if c == "'":
                j = s.index("'", i + 1)
                r += s[i + 1:j]
                i = j + 1
            elif c == "$" and i + 1 < len(s) and s[i + 1] == "{":
                j = s.index("}", i)
                inner = s[i + 2:j]
                name, _, default = inner.partition(":-")
                r += env.get(name) or (default if ":-" in inner else "")
                i = j + 1
            elif c == "$":
                j = i + 1
                while j < len(s) and (s[j].isalnum() or s[j] == "_"):
                    j += 1
                r += env.get(s[i + 1:j], "")
                i = j
            else:
                r += c
                i += 1
        out.append(r)
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Split on single-quoted parts; re.sub elsewhere.

```python
def expand(strings, env):
    rx = re.compile(r"\$\{(\w+)(:-([^}]*))?\}|\$(\w+)")
    def sub(m):
        if m.group(4):
            return env.get(m.group(4), "")
        return env.get(m.group(1)) or (m.group(3) if m.group(2) else "")
    return ["".join(p[1:-1] if p.startswith("'") else rx.sub(sub, p) for p in re.split(r"('[^']*')", s)) for s in strings]
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A character-by-character scanner. That costs O(n · L) time and O(L) space."

"I can do better: Split on single-quoted parts; re.sub elsewhere. That's O(n · L) time and O(L) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `strings = ['$A-${B:-x}'], env = {'A': '1'}`

1. sub() returns '1'
2. sub() returns 'x'
3. expand() returns ['1-x']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/expand-shell-variables)

<a id="q19"></a>
### Q19. Kill a Process Tree

🟡 Interview standard · *Children first*

Processes are `(pid, ppid, command)`. To stop `root` cleanly, signal its descendants before it: return the pids in post-order (each child's subtree in ascending pid order, then the parent).

**Examples**

- `procs = [(1, 0, 'systemd'), (100, 1, 'gunicorn master'), (101, 100, 'gunicorn worker'), (102, 100, 'gunicorn worker'), (150, 101, 'python child'), (200, 1, 'nginx'), (201, 200, 'nginx worker')], root = 100` → `[150, 101, 102, 100]`

In [ ]:
def kill_order(procs, root):
    """
    Processes are (pid, ppid, command). To stop root cleanly, signal its
    descendants before it: return the pids in post-order (each child's subtree
    in ascending pid order, then the parent).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(kill_order, [
    (([(1, 0, 'systemd'), (100, 1, 'gunicorn master'), (101, 100, 'gunicorn worker'), (102, 100, 'gunicorn worker'), (150, 101, 'python child'), (200, 1, 'nginx'), (201, 200, 'nginx worker')], 100), [150, 101, 102, 100]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursively scan the whole table for children.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Build a children map once.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Recursively scan the whole table for children.

```python
def kill_order(procs, root):
    out = []
    def visit(pid):
        kids = sorted(p for p, pp, _ in procs if pp == pid)
        for k in kids:
            visit(k)
        out.append(pid)
    visit(root)
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Build a children map once.

```python
def kill_order(procs, root):
    kids = defaultdict(list)
    for p, pp, _ in procs:
        kids[pp].append(p)
    def visit(pid):
        for k in sorted(kids[pid]):
            yield from visit(k)
        yield pid
    return list(visit(root))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursively scan the whole table for children. That costs O(n²) time and O(n) space."

"I can do better: Build a children map once. That's O(n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `procs = [(1, 0, 'a'), (2, 1, 'b')], root = 1`

1. visit() returns 2
2. visit() returns 2
3. visit() returns None
4. StopIteration raised: 
5. visit() returns 1
6. visit() returns None
7. kill\_order() returns [2, 1]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/kill-a-process-tree)

<a id="q20"></a>
### Q20. Split a Command Line Like the Shell

🟡 Interview standard · *Quotes and escapes*

Split each command into arguments the way a POSIX shell does: spaces separate arguments, single quotes keep everything literal, double quotes keep spaces but allow `\\"`, and a backslash outside quotes escapes the next character. The optimal answer is `shlex.split`.

**Examples**

- `lines = ["git commit -m 'fix: handle empty cart'", 'grep -r "TODO: later" src/', "echo it\\'s fine", 'python -c "print(\\"hi\\")"', "cp my\\ file.txt  'backup dir/'"]` → `[['git', 'commit', '-m', 'fix: handle empty cart'], ['grep', '-r', 'TODO: later', 'src/'], ['echo', "it's", 'fine'], ['python', '-c', 'print("hi")'], ['cp', 'my file.txt', 'backup dir/']]`

In [ ]:
import shlex

def split_command(lines):
    """
    Split each command into arguments the way a POSIX shell does: spaces
    separate arguments, single quotes keep everything literal, double quotes
    keep spaces but allow \\", and a backslash outside quotes escapes the next
    character. The optimal answer is shlex.split.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(split_command, [
    ((["git commit -m 'fix: handle empty cart'", 'grep -r "TODO: later" src/', "echo it\\'s fine", 'python -c "print(\\"hi\\")"', "cp my\\ file.txt  'backup dir/'"],), [['git', 'commit', '-m', 'fix: handle empty cart'], ['grep', '-r', 'TODO: later', 'src/'], ['echo', "it's", 'fine'], ['python', '-c', 'print("hi")'], ['cp', 'my file.txt', 'backup dir/']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A small state machine.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`shlex.split`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A small state machine.

```python
def split_command(lines):
    out = []
    for s in lines:
        args, cur, i, quote, started = [], "", 0, None, False
        while i < len(s):
            c = s[i]
            if quote == "'":
                if c == "'":
                    quote = None
                else:
                    cur += c
            elif quote == '"':
                if c == "\\" and i + 1 < len(s) and s[i + 1] in '"\\$`':
                    cur += s[i + 1]
                    i += 1
                elif c == '"':
                    quote = None
                else:
                    cur += c
            elif c == "\\" and i + 1 < len(s):
                cur += s[i + 1]
                started = True
                i += 1
            elif c in "'\"":
                quote, started = c, True
            elif c.isspace():
                if started:
                    args.append(cur)
                cur, started = "", False
            else:
                cur += c
                started = True
            i += 1
        if started:
            args.append(cur)
        out.append(args)
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `shlex.split`.

```python
def split_command(lines):
    return [shlex.split(s) for s in lines]
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A small state machine. That costs O(n · L) time and O(L) space."

"I can do better: `shlex.split`. That's O(n · L) time and O(L) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cli-linux/split-a-command-line-like-the-shell)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What does chmod 755 mean?</b></summary>

Owner can read, write and execute (7 = 4+2+1); group and others can read and execute (5 = 4+1). Typical for scripts and directories.

</details>

<details>
<summary><b>2. What's the difference between &gt; and &gt;&gt;?</b></summary>

`>` overwrites the file with the output, `>>` appends. `2>&1` sends errors to the same place as normal output.

</details>

<details>
<summary><b>3. What's an exit code?</b></summary>

A number a program returns when it ends: 0 for success, anything else for failure. Shells, CI systems and orchestrators use it to decide what happens next.

</details>

<details>
<summary><b>4. SIGTERM vs SIGKILL?</b></summary>

SIGTERM asks a process to stop and can be handled to clean up; SIGKILL stops it immediately and can't be caught. Always try SIGTERM first.

</details>

<details>
<summary><b>5. Why do environment variables matter?</b></summary>

They configure programs without changing code — paths, credentials, feature switches — and child processes inherit them. `export NAME=value` sets one for the session.

</details>

<details>
<summary><b>6. How do you debug a slow server?</b></summary>

Check CPU and memory (`top`), disk (`df -h`, `iostat`), open connections (`ss`), recent errors in logs (`tail`, `grep`), and what changed recently.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Moving around and reading files
- [ ] I can explain: Pipes: small tools, combined
- [ ] I can explain: Writing your own command-line tools
- [ ] I can explain: Processes, signals and scheduled jobs

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) When Does the Cron Job Run?
- [ ] [Q17](#q17) Parse an Access Log
- [ ] [Q18](#q18) Expand Shell Variables
- [ ] [Q19](#q19) Kill a Process Tree
- [ ] [Q20](#q20) Split a Command Line Like the Shell

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.